<a href="https://colab.research.google.com/github/tamayowang-lab/ML/blob/main/EEG_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import re
import os
import pickle
import numpy as np
import pandas as pd

from joblib import Parallel, delayed
from statistics import mode

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit,
                                     StratifiedGroupKFold, GridSearchCV)
from sklearn.feature_selection import SelectFromModel, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             roc_auc_score, roc_curve)


# =============================================================================
# 1. LOAD + CLEAN DATA
# =============================================================================

# INPUT DATA FORMAT
# Provide two comma-separated CSV files: baseline and six-month follow-up.
# Each file must contain:
#   - One row per participant, with exactly one observation per participant.
#   - An 'ID' or 'original_id' column containing matching participant IDs
#     across both files. Use consistent formatting (e.g., 'P001').
#   - Identical EEG feature columns containing numeric values.
#     Feature names must include one of these case-sensitive strings:
#     'AbsolutePower', 'RelativePower', 'DFA', or 'fEI'.
#   - Missing feature values represented by empty cells or NaN.
#     Each feature must have sufficient observed baseline values for imputation.
#
# Both files must contain the same participants. Row order may differ:
# observations are grouped using participant IDs, not row positions.

def load_data():

    t1 = pd.read_csv(r'C:\Users\maria\Nbt2_software\Project_folder\BUDDI_ECR\7.database\BUDDI_T1_paired_T9_yeo.csv')
    t6 = pd.read_csv(r'C:\Users\maria\Nbt2_software\Project_folder\BUDDI_ECR\7.database\BUDDI_T9_paired_T9_yeo.csv')


    for df in [t1, t6]:
        if "ID" in df.columns:
            df.rename(columns={"ID": "original_id"}, inplace=True)

    t1["Session"] = 0
    t6["Session"] = 1

    t1.columns = t1.columns.str.strip()
    t6.columns = t6.columns.str.strip()

    t1, t6 = t1.align(t6, join='outer', axis=1, fill_value=np.nan)

    data = pd.concat([t1, t6], axis=0)
    data["original_id"] = data["original_id"].astype(str)
    data = data.sort_values(["original_id", "Session"]).reset_index(drop=True)

    return data, data["Session"].values, data["original_id"].values


# =============================================================================
# 2. SESSION-AWARE PREPROCESSING
#    EEG feature groups: AbsolutePower, RelativePower, DFA, fEI
#    Scaler and imputation means fit on Session-0 (baseline) rows only.
# =============================================================================
def preprocess_data(X_train, X_test):

    exclude     = ['original_id', 'Session']
    session_col = X_train['Session'] if 'Session' in X_train.columns else None

    X_train_out = X_train[exclude].copy()
    X_test_out  = X_test[exclude].copy()

    feature_groups = ['AbsolutePower', 'RelativePower', 'DFA', 'fEI']

    for group in feature_groups:

        cols = [c for c in X_train.columns if group in c]
        if not cols:
            continue

        # fit imputation means on Session-0 training rows only
        if session_col is not None:
            baseline_mask = session_col == 0
            means = X_train.loc[baseline_mask, cols].mean()
        else:
            means = X_train[cols].mean()

        tr_filled = X_train[cols].fillna(means)
        te_filled = X_test[cols].fillna(means)

        # fit scaler on Session-0 training rows only
        if session_col is not None:
            scaler = MinMaxScaler().fit(tr_filled[baseline_mask])
        else:
            scaler = MinMaxScaler().fit(tr_filled)

        X_train_out = pd.concat([
            X_train_out,
            pd.DataFrame(scaler.transform(tr_filled),
                         columns=cols, index=X_train.index)
        ], axis=1)

        X_test_out = pd.concat([
            X_test_out,
            pd.DataFrame(scaler.transform(te_filled),
                         columns=cols, index=X_test.index)
        ], axis=1)

    return X_train_out, X_test_out


# =============================================================================
# 3. TRAIN ONE FOLD
# =============================================================================
def train_fold(train_idx, test_idx, data, y, groups, settings, seed):

    X_train = data.iloc[train_idx].copy()
    X_test  = data.iloc[test_idx].copy()
    y_train = y[train_idx]
    y_test  = y[test_idx]
    g_train = groups[train_idx]

    # ── session-aware preprocessing ────────────────────────────────────────
    X_train, X_test = preprocess_data(X_train, X_test)
    X_train = X_train.drop(columns=['original_id', 'Session'], errors='ignore')
    X_test  = X_test.drop(columns=['original_id', 'Session'], errors='ignore')

    # ── SFM: group-aware split
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=int(seed))
    base_idx, _ = next(gss.split(X_train, y_train, groups=g_train))

    X_base = X_train.iloc[base_idx]
    y_base = y_train[base_idx]

    sfm = SelectFromModel(
        LogisticRegression(solver='liblinear', max_iter=2000,
                           random_state=int(seed)),
        threshold='median'
    )
    sfm.fit(X_base, y_base)

    X_train_sfm = sfm.transform(X_train)
    X_test_sfm  = sfm.transform(X_test)
    sfm_cols    = X_train.columns[sfm.get_support()]

    # ── inner CV splitter ──────────────────────────────────────────────────
    inner_cv = StratifiedGroupKFold(n_splits=settings['CV'])

    # ── RFE sweep
    # ──────────────────────────────────────────────────────────────────────
    max_feats = X_train_sfm.shape[1]

    if max_feats >= 10:
        n_range = list(range(2, 20, 2)) + list(range(20, 151, 5))
        n_range = [n for n in n_range if n < max_feats]
        n_range.append(max_feats)
    else:
        n_range = list(range(1, max_feats + 1))

    rfe_rank = RFE(
        LogisticRegression(solver='liblinear', max_iter=2000,
                           random_state=int(seed)),
        n_features_to_select=1, step=1
    ).fit(X_train_sfm, y_train).ranking_

    C_GRID = list(np.logspace(-3, 1, 7))

    rfe_scores = []

    for n in n_range:

        X_tr = X_train_sfm[:, rfe_rank <= n]

        clf = GridSearchCV(
            LogisticRegression(solver='liblinear', max_iter=2000,
                               random_state=int(seed)),
            param_grid={'C': C_GRID},
            scoring='roc_auc',
            cv=inner_cv.split(X_tr, y_train, g_train),
            n_jobs=-1
        )
        clf.fit(X_tr, y_train)

        # score each n with the INNER-CV AUC (training data only)
        auc = clf.best_score_
        rfe_scores.append((n, auc))

    best_n = max(rfe_scores, key=lambda x: x[1])[0]

    # ── final model: top best_n features + tuned C ─────────────────────────
    final_support = rfe_rank <= best_n

    X_tr = X_train_sfm[:, final_support]
    X_te = X_test_sfm[:, final_support]

    final_clf = GridSearchCV(
        LogisticRegression(solver='liblinear', max_iter=2000,
                           random_state=int(seed)),
        param_grid={'C': C_GRID},
        scoring='roc_auc',
        cv=inner_cv.split(X_tr, y_train, g_train),
        n_jobs=-1
    )
    final_clf.fit(X_tr, y_train)

    prob = final_clf.predict_proba(X_te)[:, 1]

    fpr, tpr, thr = roc_curve(y_test, prob)
    best_thr = thr[np.argmax(tpr - fpr)]
    pred     = (prob >= best_thr).astype(int)

    final_features = sfm_cols[final_support]
    coef_abs       = np.abs(final_clf.best_estimator_.coef_[0])

    return (
        accuracy_score(y_test, pred),
        balanced_accuracy_score(y_test, pred),
        roc_auc_score(y_test, prob),
        prob,
        y_test,
        test_idx,
        list(final_features),
        list(coef_abs),
        rfe_scores,
        {'C': final_clf.best_params_['C'],
         'solver': 'liblinear', 'max_iter': 2000},
        fpr, tpr, thr
    )


# =============================================================================
# 4. NESTED CV
# =============================================================================
def nested_cv(data, y, groups, settings):

    all_probs, all_true       = [], []
    all_features, all_imp     = [], []
    all_rfe_scores            = []
    all_fpr, all_tpr, all_thr = [], [], []
    per_auc, per_bal          = [], []
    all_params                = []

    for r in range(settings['repeats']):

        seed  = np.random.default_rng().integers(0, 10**9)
        outer = GroupKFold(n_splits=settings['outer_folds'])

        results = Parallel(n_jobs=-1)(
            delayed(train_fold)(tr, te, data, y, groups, settings, seed)
            for tr, te in outer.split(data, groups=groups)
        )

        (acc, bal, auc, prob, true, idx,
         feats, imps, rfe_sc, params,
         fpr, tpr, thr) = zip(*results)

        per_auc.append(np.mean(auc))
        per_bal.append(np.mean(bal))

        p_full = np.empty(len(data))
        y_full = np.empty(len(data))
        for i, t_idx in enumerate(idx):
            p_full[t_idx] = prob[i]
            y_full[t_idx] = true[i]

        all_probs.append(p_full)
        all_true.append(y_full)

        all_features.extend(feats)
        all_imp.extend(imps)
        all_rfe_scores.extend(rfe_sc)

        all_fpr.extend(fpr)
        all_tpr.extend(tpr)
        all_thr.extend(thr)
        all_params.extend(params)

    # aggregate hyperparameters
    best_params = {}
    for k in all_params[0].keys():
        vals = [d[k] for d in all_params]
        try:
            best_params[k] = mode(vals)
        except Exception:
            best_params[k] = vals[0]

    # output dict
    return {
        'method':                          'LogisticRegression',
        'all_predicted_probs':             all_probs,
        'all_true_labels':                 all_true,
        'average_roc_auc':                 np.mean(per_auc),
        'average_outer_balanced_accuracy': np.mean(per_bal),
        'per_repeat_roc_auc':              per_auc,
        'per_repeat_balanced_acc':         per_bal,
        'best_hyperparameters':            best_params,
        'selected_features':               all_features,
        'feature_importances':             all_imp,
        'all_rfe_scores':                  all_rfe_scores,
        'all_fpr':                         all_fpr,
        'all_tpr':                         all_tpr,
        'all_thresholds':                  all_thr,
    }


# =============================================================================
# 5. RUN
# =============================================================================
data, y, groups = load_data()

settings = {
    'outer_folds': 10,
    'CV':          10,
    'repeats':     30
}

results = nested_cv(data, y, groups, settings)

# add original_id for downstream pipeline compatibility
results['original_id'] = data['original_id'].tolist()

# =============================================================================
# 6. PRINT RESULTS
# =============================================================================
print("\n=== PER REPEAT ===")
for i, (a, b) in enumerate(zip(results['per_repeat_roc_auc'],
                                results['per_repeat_balanced_acc'])):
    print(f"  Repeat {i+1:2d}: AUC={a:.3f}  BA={b:.3f}")

print("\n=== OVERALL ===")
print(f"  AUC: {np.mean(results['per_repeat_roc_auc']):.3f}"
      f" ± {np.std(results['per_repeat_roc_auc']):.3f}")
print(f"  BA : {np.mean(results['per_repeat_balanced_acc']):.3f}"
      f" ± {np.std(results['per_repeat_balanced_acc']):.3f}")